In [ ]:
import os
from pathlib import Path
root = str(Path.cwd().parent.parent)
import sys
sys.path.append(f'{root}/src')

import numpy as np
from scipy.interpolate import RegularGridInterpolator
import constants as cs
import netCDF4 as nc
import xarray as xr
from pygmt.params import Box, Position
import pyproj
import geopandas as gpd
import pygmt

In [ ]:
rsl_g = xr.open_dataset(f'{root}/inputs/rsl/rsl-CISLGM_icepc_022226_highres_512-prem.l50.ump2.lm3_lvz_200km.yt.mat-g.nc')
rsl_sg = xr.open_dataset(f'{root}/inputs/rsl/rsl-CISLGM_icepc_022226_highres_512-prem.l50.ump2.lm3_lvz_200km.yt.mat-sg.nc')
rsl_phi = xr.open_dataset(f'{root}/inputs/rsl/rsl-CISLGM_icepc_022226_highres_512-prem.l50.ump2.lm3_lvz_200km.yt.mat-phi.nc')
rsl_r = xr.open_dataset(f'{root}/inputs/rsl/rsl-CISLGM_icepc_022226_highres_512-prem.l50.ump2.lm3_lvz_200km.yt.mat-r.nc')

In [ ]:
fig = pygmt.Figure()
nrow = 1
ncol =3
w = 15
h=4

### same region as main text
xmin = 360 - 146.
xmax = 360 - 120.
ymin = 46.
ymax = 62.
region = [xmin,xmax,ymin,ymax]

time = [26, 17]

x_text = xmin #+ 0.02 * (xmax - xmin)
y_text = ymax #- 0.02 * (ymax - ymin)

### projection
projection = f'M0/0/{w/ncol*.75}c'

with fig.subplot(nrows=nrow, ncols=ncol, figsize=(f"{w}c"), frame="lrtb"):
    for i in np.arange(nrow):
        for j in np.arange(ncol):
            ind = i*ncol + j
            with fig.set_panel(panel=ind):
                pygmt.makecpt(cmap="SCM/vik+h", series=[-200, 400,1])
                ### plotting rsl, based on pannel
                if ind == 0:
                    rsl_start = rsl_g['rsl'].sel(t=time[0]) + rsl_sg['rsl'].sel(t=time[0]) + rsl_r['rsl'].sel(t=time[0]) + rsl_phi['rsl'].sel(t=time[0])
                    rsl_stop = rsl_g['rsl'].sel(t=time[-1]) + rsl_sg['rsl'].sel(t=time[-1]) + rsl_r['rsl'].sel(t=time[-1]) + rsl_phi['rsl'].sel(t=time[-1])
                    drsl = rsl_stop - rsl_start

                    fig.grdimage(grid = drsl, region=region, projection = projection)
                    ### plotting coastline
                    fig.coast(shorelines='1/0.5p,black', resolution = 'low',region=region, projection=projection, frame = ['af', 'WSne'])
                    
                    fig.text(x = x_text, y = y_text, text='A. @~\104@~RSL', fill="white", justify = 'TL', pen="0.5p,black", region=region, projection=projection)
                elif ind == 1:
                    # g and phi
                    rsl_start = rsl_g['rsl'].sel(t=time[0]) + rsl_sg['rsl'].sel(t=time[0]) + rsl_phi['rsl'].sel(t=time[0])
                    rsl_stop = rsl_g['rsl'].sel(t=time[-1]) + rsl_sg['rsl'].sel(t=time[-1]) + rsl_phi['rsl'].sel(t=time[-1])
                    drsl = rsl_stop - rsl_start

                    fig.grdimage(grid = drsl, region=region, projection = projection)
                    ### plotting coastline
                    fig.coast(shorelines='1/0.5p,black', resolution = 'low',region=region, projection=projection, frame = ['af', 'WSne'])
                    
                    fig.text(x = x_text, y = y_text, text='B. @~\104@~G & @~\104\146@~ ', fill="white", justify = 'TL', pen="0.5p,black", region=region, projection=projection)

                    fig.colorbar(frame =['a200f50', 'x+l@~\104@~RSLRelative Sea Level from 26 ka to 17 ka (m)'],
                                position=Position("BC", 'outside', offset =(-0.5,-6), anchor='MC'),
                                orientation='horizontal', length = '100%')
                elif ind == 2:
                    # r only
                    rsl_start = rsl_r['rsl'].sel(t=time[0]) 
                    rsl_stop = rsl_r['rsl'].sel(t=time[-1]) 
                    drsl = rsl_stop - rsl_start

                    fig.grdimage(grid = drsl, region=region, projection = projection)
                    
                    ### plotting coastline
                    fig.coast(shorelines='1/0.5p,black', resolution = 'low',region=region, projection=projection, frame = ['af', 'WSne'])
                    fig.text(x = x_text, y = y_text, text='C. @~\104@~R',fill="white", justify = 'TL', pen="0.5p,black", region=region, projection=projection)


fig.show()
# fig.savefig(f'{root}/figures/sf_RSL_decomp.pdf')